# Alakoro + DASDAE: Integração DASCore e Xdas

Este notebook demonstra a integração do Alakoro com o ecossistema DASDAE: DASCore (Patch/Spool) e Xdas (DataArray/DataCollection), incluindo leitura/escrita de formatos e pipeline híbrido com processadores C++20.

In [ ]:
import numpy as np
import dascore as dc

from src.io.alakoro_spool import AlakoroPatch, AlakoroSpool
from src.io.dasdae import DASDAEAdapter
from src.io.dascore_formats import read as read_dascore, write as write_dascore, supported_formats as supported_dascore_formats
from src.io.xdas_formats import read_xdas, write_xdas, supported_xdas_formats
from src.io.xdas_adapter import alakoro_to_xdas, spool_to_datacollection
from src.processing.hybrid_pipeline import HybridPipeline

## 1. DASCore — Conversão básica Alakoro ↔ Patch/Spool

In [ ]:
# Criar dados sintéticos
data = np.random.randn(500, 32)
patch = DASDAEAdapter.array_to_patch(data, dt_s=1.0, dx_m=2.0, modality='das')
alakoro = AlakoroPatch(patch, well_id='BRA-001')
alakoro

In [ ]:
# Operações DASCore via AlakoroPatch
detrended = alakoro.detrend()
filtered = detrended.pass_filter((0.5, 25.0))
filtered.shape

In [ ]:
# Spool
spool = AlakoroSpool([alakoro, detrended, filtered])
spool.get_contents()

## 2. DASCore — Leitura e escrita de formatos

In [ ]:
# Formatos detectados no DASCore
supported_dascore_formats()[:10]

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    # Escrever em formato DASDAE
    path = Path(tmp) / 'example.dasdae'
    write_dascore(alakoro, path)
    back = read_dascore(path, well_id='BRA-001')
    print(back, back.shape, np.allclose(back.data, data))

    # Escrever em pickle (preserva spools)
    spool_path = Path(tmp) / 'spool.pickle'
    write_dascore(spool, spool_path)
    back_spool = read_dascore(spool_path, well_id='BRA-001')
    print(back_spool, len(back_spool))

## 3. Xdas — Conversão direta Alakoro ↔ DataArray/DataCollection

In [ ]:
# AlakoroPatch -> xdas.DataArray
da = alakoro_to_xdas(alakoro)
print(da.shape, da.attrs)

# AlakoroSpool -> xdas.DataCollection
collection = spool_to_datacollection(spool)
print('Keys:', list(collection.keys()))

## 4. Xdas — Leitura e escrita de formatos

In [ ]:
# Formatos detectados no Xdas
supported_xdas_formats()[:10]

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    # NetCDF roundtrip
    path = Path(tmp) / 'example.nc'
    write_xdas(alakoro, path)
    back = read_xdas(path, well_id='BRA-001')
    print(back, back.shape, np.allclose(back.data, data))

    # Múltiplos arquivos
    p1 = Path(tmp) / 'p1.nc'
    p2 = Path(tmp) / 'p2.nc'
    write_xdas(alakoro, p1)
    write_xdas(alakoro, p2)
    multi = read_xdas([p1, p2], well_id='BRA-001')
    print('multi:', multi.shape)

## 5. Pipeline híbrido DASCore/Xdas + C++20

In [ ]:
pipeline = (
    HybridPipeline(alakoro)
    .dascore('detrend', dim='time', type='linear')
    .dascore('pass_filter', time=(0.5, 25.0))
    .xdas('filter', freq=0.2, dim='time', btype='lowpass')
    .cpp('median_filter_1d', window_size=5)
    .dascore('decimate', time=2)
)
result = pipeline.to_patch()
print(result)
print('Passos:', pipeline.history)

In [ ]:
# Processadores que retornam arrays (psd, cwt, rfft)
psd = HybridPipeline(alakoro).apply_array('psd', sample_rate_hz=1.0)
rfft = HybridPipeline(alakoro).apply_array_xdas('rfft')
print('PSD shape:', psd.shape)
print('RFFT shape:', rfft.shape)